In [ ]:
def table(period, mean_curve, op_cost, op_cost_rate, mean_price_int, fixed_cost, profit, dr, init_invest):
    """
    BDH-style table from t = 0 ... period (inclusive).
    Leading 'PV' column shows PV at t = -1.
    No 'Remaining reserves' row and no 'Cash flow payout rate' row.

    Display formatting:
      - All cells: 1 decimal place
      - Row 'Production level': 3 decimal places
      - 'PV' column shows PV(t=-1) only on 'PV of cash flows' with 1 decimal
    """
    import numpy as np
    import pandas as pd

    # ---- Validate and build series ----
    n = period + 1                      # t = 0..period
    cols_time = [str(i) for i in range(n)]

    prod = np.asarray(mean_curve, dtype=float).reshape(-1)
    if len(prod) != n:
        raise ValueError("mean_curve must have length = period+1 (t=0..period).")

    if isinstance(op_cost, (int, float)):
        voc_series = np.array([float(op_cost) * (1 + op_cost_rate) ** t for t in range(n)], dtype=float)
    else:
        voc_series = np.asarray(op_cost, dtype=float).reshape(-1)
        if len(voc_series) != n:
            raise ValueError("op_cost series must have length = period+1.")

    if isinstance(mean_price_int, (int, float)):
        price_series = np.full(n, float(mean_price_int), dtype=float)
    else:
        price_series = np.asarray(mean_price_int, dtype=float).reshape(-1)
        if len(price_series) != n:
            raise ValueError("mean_price_int series must have length = period+1.")

    # ---- Table rows ----
    rows = [
        'Production level', 'Variable op cost rate', 'Oil price',
        'Revenues', 'Production cost', 'Cash flow', 'Profit sharing',
        'Net cash flows', 'PV of cash flows'
    ]
    data = np.zeros((len(rows), n), dtype=float)

    # Fill rows (all numeric)
    data[0, :] = prod                                   # Production level
    data[1, :] = voc_series                             # Variable op cost rate
    data[2, :] = price_series                           # Oil price
    data[3, :] = data[2, :] * data[0, :]               # Revenues
    data[4, :] = data[1, :] * data[0, :] + fixed_cost  # Production cost
    data[5, :] = data[3, :] - data[4, :]               # Cash flow
    data[6, :] = data[5, :] * profit                   # Profit sharing
    data[7, :] = data[5, :] - data[6, :]               # Net cash flows

    # PV_t = sum_{j=t..period} Net[j] / (1+dr)^{j-t}
    for t in range(n - 1, -1, -1):
        expnts = np.arange(t, n) - t
        disc = (1 + dr) ** expnts
        data[8, t] = float(np.sum(data[7, t:] / disc))  # PV of cash flows at time t

    # --- Values at t = -1 ---
    pv_t0 = float(data[8, 0])          # PV at t=0
    # market_val = pv_t0 / (1 + dr)      # PV at t=-1
    market_val = pv_t0
    NPV = market_val - init_invest

    # ---- DataFrame (numeric) ----
    frame_num = pd.DataFrame(data, index=rows, columns=cols_time)

    # ---- Build display DataFrame with custom formatting ----
    frame_disp = frame_num.copy()

    # Insert leading 'PV' column: strings for display
    #frame_disp.insert(0, '', [''] * len(rows))
    #frame_disp.at['PV of cash flows', 'PV'] = f"{market_val:.2f}"

    # Format all time columns with 1 decimal
    for c in cols_time:
        frame_disp[c] = frame_disp[c].map(lambda x: f"{float(x):.2f}")

    # Override 'Production level' row with 3 decimals
    for c in cols_time:
        frame_disp.at['Production level', c] = f"{frame_num.at['Production level', c]:.3f}"

    # Cosmetic: remove column axis name
    frame_disp = frame_disp.rename_axis('', axis='columns')

    return frame_disp, market_val, float(NPV)
